In [2]:
import os, subprocess, sys
#local imports
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append( os.path.abspath(pymipl_path+'/xnat_workflow') ) 
from dicom_sort import *
from pathlib import Path


In [8]:
import csv
#read the scan type mappings.
with open('/workspace/mmilchenko/BM_WU/washu_index_four_mod.csv') as f:
    sessions=list(csv.DictReader(f))

In [6]:
sessions[0]

{'Subject': '0',
 'Session Label': 'M79614744',
 'id_t2f': 'M79614744_20201110144849',
 'series_description_t2f': '4',
 'frames_t2f': 'TRA FLAIR NEW',
 'id_t1ce': '27',
 'series_description_t1ce': '17',
 'frames_t1ce': 'TRA 3D T1 GRE++STRAIGHT++_ND',
 'id': '192',
 'series_description': '2',
 'frames': 'SAG T1',
 'id_t2w': '25',
 'series_description_t2w': '3',
 'frames_t2w': 'TRA T2',
 None: ['27']}

In [7]:
#define global variables and helper functions. 
from workflow_adapters import workflow_to_batch, init_global_vars_bootstrap_image, sync_resource_xnat
import logging
import datetime
import yaml

def set_logger():
    root = logging.getLogger()
    root.setLevel(logging.DEBUG)
    
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    root.addHandler(handler)    

#helper functions
def paths_to_str(x):
    if isinstance(x, Path): return str(x)
    if isinstance(x, dict): return {k: paths_to_str(v) for k, v in x.items()}
    if isinstance(x, list): return [paths_to_str(v) for v in x]
    return x

def resource_to_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment):
    return sync_resource_xnat(local_resource, xnat_session_resource, xnat_project, xnat_subject, xnat_experiment, \
        level="experiment", create_hierarchy=True)


global_vars={}
env_type='jupyter'
#env_type='container'
project='BM_WU'
workflow_id='BRATS_preproc'
global_vars['g_workflow_id']=workflow_id
root_dir=Path('/workspace/mmilchenko')

if env_type=='jupyter':
    pymipl_dir=root_dir / "pymipl"

    #path that mounts directory with XNAT experiments
    input_mount_path=Path('/data/projects/BM_WU')
    local_workdir_path=Path('/workdir/mmilchenko/BM_WU')
    
    global_vars['g_input_mount_path']=input_mount_path
    #path to local directory where processing will be stored
    global_vars['g_local_workdir_path']=local_workdir_path
    #library locations, algorithm specific
    global_vars['g_pymipl_dir']=pymipl_dir
    #main algorithm repository dir
    global_vars['g_env_repo_dir']=root_dir / 'envs/brats'
    #global_vars['g_alg_repo_dir']
    global_vars['g_project']=project
        
elif env_type == 'container': #built-in defaults used in the bootstrap image.
    init_global_vars_bootstrap_image(global_vars,project)


In [10]:
#loop over sessions to create batch scripts.
dt=datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file=local_workdir_path / f"batch_{dt}.sh"

for session in sessions:
    job,steps={},[]

    job_scan_context=global_vars['g_input_mount_path'] / Path('SCANS') #/ job_scan_id / 'DICOM'
    if env_type == 'jupyter': job_scan_context = Path(job_experiment) / job_scan_context

    step={"step_title": "Run BRATS preprocessing"}
    
    step['job_t1w_path'] = job_scan_context / session['id'] / 'DICOM'
    step['job_t1wce_path'] = job_scan_context / session['id_t1ce'] / 'DICOM'
    step['job_t2w_path'] = job_scan_context / session['id_t2w'] / 'DICOM'
    step['job_t2f_path'] = job_scan_context / session['id_t2f'] / 'DICOM'
    step['job_subject'] = session['Subject']
    step['job_experiment'] =session['Session Label']
    step['job_outdir']="{g_local_workdir_path}/{job_subject}/{job_experiment}"
    
    step['step_command']="micromamba run -p {g_env_repo_dir} {g_pymipl_dir}/brats_preprocess.py --patient_id {job_subject} \
        --outdir {job_outdir} --t1 {job_t1w_path} --t1ce {job_t1wce_path} --t2 {job_t2w_path} --flair {job_t2f_path}"

    steps+=[step]

    with open(job_file_yaml,"w") as f:
        yaml.safe_dump(paths_to_str(job),f,sort_keys=False)
        
    if env_type=='jupyter': #write all commands to a single batch file
        workflow_to_batch(job,global_vars,batch_file)
        
    else: #one batch file per job
        #create batch        
        print(job_file_sh)
        #reset job script
        ! truncate -s 0 {job_file_sh}
        #generate job script
        workflow_to_batch(job,global_vars,job_file_sh)
        ! chmod +x {job_file_sh}
        #store batch file.
        print('sending batch to xnat resource')
        res=resource_to_xnat(job_dir, job_id, project, job_subject, job_exp_label)
        if res == 0:
            print('Success')
        else: 
            print (f'Failed sending configuration to session resource, error {res}')
            
    break #debug: do just one run.